<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M02_04_Agregacion_exportacion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Introducción a la Ciencia de Datos

## Módulo 2: Organización y preparación de datos

### Notebook 4: Agregación temporal y exportación de datos

En los notebooks anteriores aprendimos a importar, explorar, organizar, seleccionar y limpiar datos. También trabajamos con fechas, índices temporales y control de calidad.

En este notebook aprenderemos a trabajar con datos registrados a diferentes **resoluciones temporales**. Partiremos de observaciones meteorológicas registradas cada **15 minutos** y las transformaremos a escalas horarias, diarias y mensuales.

Este proceso se denomina **agregación temporal** y es una operación frecuente cuando trabajamos con datos ambientales obtenidos mediante sensores automáticos.

También aprenderemos a exportar los datos procesados para utilizarlos posteriormente en otros análisis.

### Objetivos

Al finalizar este notebook podrás:

- identificar la resolución temporal de una serie;
- comprender el concepto de agregación temporal;
- utilizar `resample()` para cambiar la resolución temporal de los datos;
- transformar datos de 15 minutos a escalas horarias, diarias y mensuales;
- aplicar funciones de agregación como `sum()`, `mean()`, `min()` y `max()`;
- seleccionar la función de agregación apropiada según la variable analizada;
- exportar los datos procesados a archivos CSV y Excel.

## Datos utilizados: estación meteorológica CH01 – Campus Chillán UdeC

En este notebook trabajaremos con datos provenientes de la estación meteorológica **CH01**, ubicada en el Campus Chillán de la Universidad de Concepción.

La estación forma parte de un sistema de monitoreo meteorológico de bajo costo que registra automáticamente diferentes variables ambientales.

Entre las variables monitoreadas se encuentran:

- temperatura del aire;
- humedad relativa;
- presión atmosférica;
- precipitación;
- velocidad y dirección del viento;
- radiación solar;
- índice UV.

Los datos de la estación pueden visualizarse en tiempo real en la plataforma **MeteoChillán UdeC**.

**Red de estaciones:**  
https://meteochillan.udec.cl/stations.html

**Estación CH01 – Campus Chillán:**  
https://meteochillan.udec.cl/stations/ch_01.html

Para este ejercicio utilizaremos registros históricos con una **resolución temporal de 15 minutos**.

Trabajar con datos de alta frecuencia nos permitirá estudiar cómo una misma serie puede representarse a diferentes escalas temporales:

**15 minutos → horario → diario → mensual**

## 1. Resolución temporal de los datos

La **resolución temporal** indica el intervalo de tiempo entre observaciones consecutivas de una serie.

Por ejemplo, un sensor que registra información cada 15 minutos genera observaciones como:

| Fecha y hora | Temperatura (°C) |
|---|---:|
| 2026-08-10 10:00 | 8.2 |
| 2026-08-10 10:15 | 8.6 |
| 2026-08-10 10:30 | 9.1 |
| 2026-08-10 10:45 | 9.4 |
| 2026-08-10 11:00 | 9.8 |

En este caso, la resolución temporal de los datos es de **15 minutos**.

Una alta resolución temporal permite observar cambios que ocurren dentro de un mismo día. Sin embargo, dependiendo de la pregunta que queremos responder, puede ser más conveniente trabajar con información horaria, diaria o mensual.

Por ejemplo:

- **15 minutos:** ¿cómo cambia la temperatura durante la mañana?
- **horaria:** ¿cuál fue la temperatura media de cada hora?
- **diaria:** ¿cuál fue la temperatura media, mínima o máxima de cada día?
- **mensual:** ¿cómo cambió la temperatura media entre diferentes meses?

La resolución temporal utilizada depende, por lo tanto, del objetivo del análisis.

## 2. Agregación temporal

La **agregación temporal** consiste en agrupar varias observaciones correspondientes a un determinado intervalo de tiempo y resumirlas mediante una operación.

Por ejemplo, nuestra estación registra cuatro valores de temperatura durante una hora:

| Fecha y hora | Temperatura (°C) |
|---|---:|
| 10:00 | 8.2 |
| 10:15 | 8.6 |
| 10:30 | 9.1 |
| 10:45 | 9.4 |

Estas cuatro observaciones pueden utilizarse para obtener un único valor que represente esa hora.

Por ejemplo, podemos calcular la **temperatura media horaria**.

De esta manera:

**4 observaciones de 15 minutos → 1 observación horaria**

El mismo procedimiento puede aplicarse posteriormente para obtener datos diarios o mensuales:

**15 minutos → horario → diario → mensual**

Al agregar los datos disminuye el número de observaciones y cambia la información representada por cada valor.

## 3. La operación depende de la variable

Agregar datos no significa simplemente cambiar su frecuencia temporal. También debemos decidir **qué operación utilizar para resumir las observaciones**.

La operación apropiada depende de la variable y de la pregunta que queremos responder.

### Temperatura

A partir de observaciones de temperatura cada 15 minutos podemos calcular, por ejemplo:

- `mean()` → temperatura media del periodo;
- `min()` → temperatura mínima del periodo;
- `max()` → temperatura máxima del periodo.

### Precipitación

Si los registros representan la precipitación acumulada durante cada intervalo, podemos utilizar:

- `sum()` → precipitación total acumulada durante el periodo.

Por ejemplo:

**Temperatura cada 15 min → `mean()` → temperatura media horaria**

**Precipitación cada 15 min → `sum()` → precipitación acumulada horaria**

Por lo tanto, antes de realizar una agregación debemos preguntarnos:

1. ¿Qué representa cada observación original?
2. ¿Qué periodo queremos representar?
3. ¿Qué operación tiene sentido para esa variable?

Python puede ejecutar diferentes operaciones matemáticas, pero **no todas tienen necesariamente una interpretación adecuada para la variable analizada**.

## 4. Cargar y explorar los datos de la estación CH01

Ahora trabajaremos con los registros reales de la estación CH01.

Antes de realizar cualquier agregación debemos conocer la estructura del archivo y comprobar:

- qué columnas contiene;
- cómo están almacenadas la fecha y la hora;
- qué variables meteorológicas están disponibles;
- qué unidades utilizan;
- cuántos registros contiene la serie;
- cuál es el periodo disponible.

Primero importaremos las librerías necesarias.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

### Descargar los datos

Trabajaremos con el archivo **`AW005_2026_08.csv`**, que contiene registros de alta frecuencia de una estación meteorológica de bajo costo.

Descarga el archivo desde el siguiente enlace:

[**Descargar AW005_2026_08.csv**](https://github.com/paulmunozpauta/VLIR_SI_EC_2025-2027/blob/main/datasets/AW005_2026_08.csv)

Una vez descargado, guarda el archivo en tu computador. A continuación lo cargaremos manualmente en Google Colab.

### Cargar el archivo en Google Colab

Los archivos almacenados en nuestro computador no están disponibles automáticamente en Google Colab.

Utilizaremos `files.upload()` para seleccionar y cargar el archivo que acabamos de descargar.

Al ejecutar la siguiente celda:

1. selecciona `AW005_2026_08.csv`;
2. espera hasta que termine la carga;
3. comprueba que aparezca el nombre del archivo.

In [ ]:
from google.colab import files

uploaded = files.upload()

### Leer el archivo con Pandas

El archivo se encuentra en formato **CSV** (*Comma-Separated Values*).

Como hemos visto anteriormente, Pandas permite importar este tipo de archivos mediante la función:

`pd.read_csv()`

Guardaremos el contenido en un DataFrame llamado `datos`.

In [ ]:
datos = pd.read_csv("AW005_2026_08.csv")

datos.head()

`head()` muestra las primeras cinco filas del DataFrame y nos permite realizar una primera inspección de los datos.

Cada fila corresponde a un registro de la estación meteorológica.

Observa:

- la columna que contiene la fecha y hora;
- las variables meteorológicas disponibles;
- los nombres utilizados para las columnas;
- la forma en que están representados los valores.

Antes de realizar cualquier agregación temporal, debemos conocer y preparar correctamente la estructura temporal de estos datos.

### Dimensiones y variables disponibles

Antes de concentrarnos en la dimensión temporal, revisaremos brevemente el tamaño del conjunto de datos y las variables disponibles.

Recordemos que `shape` devuelve:

`(número de filas, número de columnas)`

In [ ]:
datos.shape

El archivo contiene una gran cantidad de observaciones porque la estación registra información con una frecuencia temporal alta.

Esto es una característica habitual de los sistemas automáticos de monitoreo: incluso periodos relativamente cortos pueden generar miles de registros.

Ahora revisaremos los nombres de las columnas.

In [ ]:
datos.columns


Cada columna representa una variable registrada o calculada por el sistema.

Para este notebook nos concentraremos inicialmente en dos elementos:

- `timestamp`: fecha y hora asociada a cada observación;
- `temperature_C`: temperatura del aire en grados Celsius.

Antes de utilizarlas debemos comprobar cómo fueron interpretadas por Pandas al importar el archivo.

In [ ]:
datos.dtypes


## 6. Preparar la fecha y hora

La columna `timestamp` contiene información fundamental para nuestro análisis: indica **cuándo fue realizada cada observación**.

Aunque visualmente observemos fechas y horas, esto no significa necesariamente que Pandas las haya reconocido como información temporal.

Revisemos específicamente el tipo de dato de esta columna.

In [ ]:
datos["timestamp"].dtype

Si el resultado es `object`, Pandas está tratando los valores de `timestamp` como texto y no como fechas y horas.

Para realizar operaciones temporales como:

- seleccionar periodos;
- ordenar cronológicamente;
- calcular intervalos entre observaciones;
- agregar datos por hora, día o mes;

necesitamos convertir esta columna al tipo `datetime`.

Utilizaremos nuevamente `pd.to_datetime()`, pero ahora trabajaremos no solamente con una fecha, sino con **fecha y hora**.

In [ ]:
datos["timestamp"] = pd.to_datetime(
    datos["timestamp"]
)

Comprobemos nuevamente el tipo de dato.

In [ ]:
datos["timestamp"].dtype

Ahora `timestamp` debería aparecer como un tipo `datetime`.

Esto permite que Pandas interprete correctamente elementos como:

- año;
- mes;
- día;
- hora;
- minuto.

Al igual que hicimos anteriormente con las series diarias de precipitación, utilizaremos la información temporal como **índice del DataFrame**.

In [ ]:
datos = datos.set_index("timestamp")

datos.head()

Ahora cada fila está identificada por la fecha y hora en que se realizó la observación.

Tener un `DatetimeIndex` será fundamental para utilizar posteriormente `resample()`.

Antes de agregar los datos debemos comprobar dos aspectos:

1. que las observaciones estén ordenadas cronológicamente;
2. cuál es realmente el intervalo temporal entre los registros.

Primero verificaremos el orden.

In [ ]:
datos.index.is_monotonic_increasing

Si obtenemos `True`, las observaciones están ordenadas desde la más antigua hasta la más reciente.

De todas formas, podemos utilizar `sort_index()` para asegurarnos de mantener el orden cronológico.

In [ ]:
datos = datos.sort_index()

## 7. ¿Cuál es la resolución temporal de los datos?

La frecuencia de registro de un sistema de monitoreo puede cambiar durante su periodo de operación.

En esta estación, los primeros registros fueron almacenados aproximadamente **cada una hora**. Posteriormente, la configuración del sistema fue modificada para registrar observaciones **cada 15 minutos**.

Además, el momento exacto en que un registro queda almacenado puede presentar pequeñas diferencias de segundos debido al proceso de transmisión y almacenamiento de los datos.

Por esta razón, antes de realizar una agregación temporal, examinaremos directamente cuánto tiempo transcurre entre observaciones consecutivas.

Podemos calcular esta diferencia utilizando `diff()` sobre el índice temporal.

In [ ]:
intervalos = datos.index.to_series().diff()

intervalos.head(10)

In [ ]:
intervalos.tail(10)

`diff()` calcula la diferencia entre cada fecha y hora y la observación inmediatamente anterior.

El primer registro aparece como `NaT` (*Not a Time*) porque no existe una observación anterior con la cual calcular la diferencia.

Si todos los datos hubieran sido registrados exactamente cada 15 minutos, esperaríamos encontrar repetidamente:

`0 days 00:15:00`

Sin embargo, estamos trabajando con datos reales y sabemos que la frecuencia de registro cambió durante el periodo de funcionamiento de la estación.

Exploremos cuáles son los intervalos más frecuentes.

In [ ]:
intervalos.value_counts().head(20)

El resultado muestra que el intervalo más frecuente es actualmente **15 minutos**.

Sin embargo, también aparecen muchos otros intervalos.

Esto ocurre por dos razones principales:

1. **Cambio en la frecuencia de registro:** durante una primera etapa, la estación registraba información aproximadamente cada hora. Posteriormente, la frecuencia se modificó a 15 minutos.

2. **Pequeñas diferencias en el tiempo de registro:** algunos registros horarios no están separados exactamente por una hora. Por ejemplo, pueden aparecer diferencias como `01:00:00.219` o `00:59:57.527`, asociadas al proceso de transmisión y almacenamiento de la información.

También pueden existir intervalos mayores, por ejemplo 30 o 45 minutos, cuando no se dispone de una observación intermedia.

Este ejemplo muestra una característica importante de los datos obtenidos mediante sistemas automáticos de monitoreo:

> **La frecuencia esperada de medición y los intervalos que observamos realmente en los datos no siempre son exactamente iguales.**

Por esta razón es importante explorar la dimensión temporal antes de realizar una agregación.

### ¿Cuándo cambió la frecuencia de registro?

Podemos visualizar los intervalos entre observaciones a lo largo del tiempo.

Para facilitar la interpretación, transformaremos cada intervalo a minutos.

In [ ]:
intervalos_min = intervalos.dt.total_seconds() / 60

intervalos_min.head()

`total_seconds()` transforma cada intervalo temporal en segundos.

Al dividir por 60 obtenemos el intervalo entre observaciones expresado en **minutos**.

Ahora podemos representarlo gráficamente.

### Visualizar el cambio en la frecuencia de registro

En los resultados anteriores observamos que algunos intervalos entre registros son muy grandes debido a periodos sin observaciones.

Si representamos todos estos intervalos en el mismo gráfico, estos valores dominan la escala vertical y dificultan observar las frecuencias de registro habituales.

Por esta razón, en el siguiente gráfico mostraremos solamente el rango entre **0 y 90 minutos**.

Esto **no elimina ni modifica los datos**. Únicamente cambia el rango visible del gráfico para facilitar la interpretación.

Además, incorporaremos líneas de referencia para:

- **60 minutos**, correspondiente aproximadamente a la frecuencia utilizada inicialmente;
- **15 minutos**, correspondiente a la frecuencia utilizada posteriormente.

In [ ]:
plt.figure(figsize=(14, 5))

plt.scatter(
    intervalos_min.index,
    intervalos_min,
    s=8,
    alpha=0.6
)

# Líneas de referencia
plt.axhline(60, linestyle="--", linewidth=1.5, label="60 minutos")
plt.axhline(15, linestyle="--", linewidth=1.5, label="15 minutos")

# Limitar el eje para observar las frecuencias principales
plt.ylim(0, 90)

plt.xlabel("Fecha")
plt.ylabel("Intervalo entre registros (minutos)")
plt.title("Intervalo temporal entre observaciones de la estación CH01")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

El gráfico permite identificar dos etapas en el funcionamiento de la estación.

Durante la primera parte del registro, las observaciones se encuentran aproximadamente separadas por **60 minutos**. Posteriormente, la frecuencia de adquisición cambia y los registros pasan a estar separados principalmente por **15 minutos**.

También podemos observar interrupciones y algunos intervalos diferentes a las frecuencias esperadas.

Este ejemplo muestra que una serie temporal real no necesariamente presenta una frecuencia perfectamente regular durante todo su periodo de registro.

En la siguiente sección utilizaremos `resample()` para agrupar las observaciones en intervalos temporales definidos.

## 8. Agregación temporal con `resample()`

Ahora que conocemos la estructura temporal de nuestros datos, podemos comenzar a trabajar con diferentes escalas de tiempo.

Pandas proporciona el método `resample()` para **agrupar una serie temporal en intervalos regulares**.

Su funcionamiento general es:

`datos.resample("frecuencia").operacion()`

Debemos indicar dos elementos:

1. **La nueva frecuencia temporal** que queremos obtener.
2. **La operación** que utilizaremos para resumir las observaciones dentro de cada intervalo.

Algunos ejemplos de frecuencias son:

- `"h"` → horario;
- `"D"` → diario;
- `"ME"` → mensual.

Y algunas operaciones que podemos aplicar son:

- `mean()` → media;
- `min()` → mínimo;
- `max()` → máximo;
- `sum()` → suma.

Comenzaremos transformando las observaciones de temperatura de la estación a una resolución **horaria**.

### Seleccionar la temperatura

Nuestro DataFrame contiene varias variables meteorológicas.

Para comprender primero el funcionamiento de `resample()`, trabajaremos únicamente con la variable de **temperatura del aire**.

Revisemos nuevamente sus primeras observaciones.

In [ ]:
temperatura = datos[["temperature_c"]].copy()

temperatura.head()

Hemos creado un nuevo DataFrame llamado `temperatura` que contiene:

- `timestamp` como índice temporal;
- `temperature_c` como variable.

No hemos modificado el DataFrame original `datos`.

Ahora podemos utilizar el índice temporal para agrupar las observaciones por hora.

### Temperatura horaria

Para obtener una temperatura representativa de cada hora utilizaremos la **media** de las observaciones disponibles dentro de esa hora.

Por ejemplo, si tenemos:

| Hora | Temperatura (°C) |
|---|---:|
| 10:00 | 8.2 |
| 10:15 | 8.6 |
| 10:30 | 9.1 |
| 10:45 | 9.4 |

las cuatro observaciones pertenecen a la misma hora.

Con:

`resample("h").mean()`

Pandas agrupa automáticamente las observaciones por hora y calcula su media.

El resultado tendrá **una fila por hora**, en lugar de una fila por cada registro original.

In [ ]:
temperatura_horaria = temperatura.resample("h").mean()

temperatura_horaria.head(10)

Observa ahora el índice del nuevo DataFrame.

Cada fila representa una **hora**, y `temperature_C` corresponde a la temperatura media calculada utilizando las observaciones disponibles dentro de esa hora.

Es importante recordar que nuestra serie original no tuvo siempre la misma frecuencia.

Durante el periodo inicial existen aproximadamente **una observación por hora**, mientras que posteriormente existen aproximadamente **cuatro observaciones por hora**.

Por lo tanto, aunque `resample()` genera intervalos horarios regulares, la cantidad de observaciones utilizadas para calcular cada media puede ser diferente.

## 9. De datos horarios a datos diarios

Ahora aumentaremos la escala temporal y construiremos una serie **diaria** de temperatura.

Para caracterizar la temperatura de un día, un único valor puede no ser suficiente. Podemos estar interesados en diferentes características:

- **temperatura media diaria**;
- **temperatura mínima diaria**;
- **temperatura máxima diaria**.

Pandas permite calcular varias funciones de agregación simultáneamente mediante `agg()`.

En este caso utilizaremos:

- `mean` → temperatura media del día;
- `min` → temperatura mínima del día;
- `max` → temperatura máxima del día.

Para definir intervalos diarios utilizaremos:

`resample("D")`

### ¿Cuántos datos necesitamos para calcular un valor diario?

`resample()` utiliza las observaciones disponibles para calcular cada valor agregado.

Esto significa que podría calcular una temperatura media diaria incluso si faltan varias horas del día.

Para evitar representar días con muy poca información, estableceremos un **criterio de completitud del 80 %**.

Un día tiene 24 horas:

**24 × 0.80 = 19.2**

Por lo tanto, consideraremos válido un valor diario solamente cuando existan al menos **19 horas con datos**.

> El 80 % es un criterio definido para este ejercicio y no constituye una regla universal. El nivel de completitud requerido depende de la variable, el objetivo del análisis y los criterios utilizados en cada estudio.

In [ ]:
# Número de horas con datos en cada día
n_horas = temperatura_horaria["temperature_c"].resample("D").count()

# Agregación diaria
temperatura_diaria = temperatura_horaria.resample("D").agg(
    ["mean", "min", "max"]
)

# Los días con menos de 22 horas disponibles se consideran incompletos (buscamos tener al menos el 90 % datos)
temperatura_diaria.loc[n_horas < 19, :] = float("nan")

temperatura_diaria.head()

El nuevo DataFrame contiene **una fila por día** y tres valores de temperatura para cada fecha:

- `mean`: temperatura media diaria;
- `min`: temperatura mínima registrada durante el día;
- `max`: temperatura máxima registrada durante el día.

Observa que el número de registros se ha reducido considerablemente respecto de la serie original.

Esto ocurre porque múltiples observaciones realizadas durante un mismo día han sido resumidas en una única fila.

### Renombrar las variables

Los nombres de las columnas generados automáticamente por `agg()` pueden resultar poco prácticos.

Para facilitar el trabajo posterior, asignaremos nombres más descriptivos:

- `temperatura_media_C`;
- `temperatura_min_C`;
- `temperatura_max_C`.

In [ ]:
temperatura_diaria.columns = [
    "temperatura_media_C",
    "temperatura_min_C",
    "temperatura_max_C"
]

temperatura_diaria.head()

### Visualizar la temperatura diaria

Ahora podemos representar conjuntamente la temperatura media, mínima y máxima diaria.

Este gráfico resume una gran cantidad de observaciones de alta frecuencia y permite observar más fácilmente la evolución de la temperatura a lo largo del tiempo.

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    temperatura_diaria.index,
    temperatura_diaria["temperatura_media_C"],
    label="Media"
)

plt.plot(
    temperatura_diaria.index,
    temperatura_diaria["temperatura_min_C"],
    label="Mínima",
    alpha=0.7
)

plt.plot(
    temperatura_diaria.index,
    temperatura_diaria["temperatura_max_C"],
    label="Máxima",
    alpha=0.7
)

plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.title("Temperatura diaria – Estación CH01")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

La agregación diaria permite observar con mayor claridad la evolución temporal de la temperatura.

Sin embargo, debemos recordar que al agregar los datos **perdemos parte del detalle temporal original**.

La serie original permite estudiar cambios dentro de cada día, mientras que la serie diaria resume todas esas observaciones mediante algunos valores representativos.

Por lo tanto, una resolución temporal más agregada no es necesariamente mejor o peor. La escala adecuada depende de la pregunta que queremos responder.

## 10. De datos diarios a datos mensuales

Podemos continuar el proceso de agregación y resumir ahora la información a escala **mensual**.

Para cada mes calcularemos:

- **temperatura media mensual**;
- **temperatura mínima registrada durante el mes**;
- **temperatura máxima registrada durante el mes**.

Para trabajar con meses utilizaremos:

`resample("ME")`

donde `ME` indica el **final de cada mes** (*Month End*).

### Criterio de completitud para la agregación mensual

Al igual que para los datos diarios, debemos comprobar que exista suficiente información antes de calcular un valor mensual.

En este ejercicio utilizaremos nuevamente un criterio de **80 % de completitud**.

Sin embargo, el número de días cambia entre meses:

- un mes de 31 días requiere al menos **25 días válidos**;
- un mes de 30 días requiere al menos **24 días válidos**;
- febrero requiere un número diferente según tenga 28 o 29 días.

En lugar de establecer manualmente estos valores, calcularemos la proporción de días válidos para cada mes.

Un mes será considerado válido solamente cuando:

**días con datos válidos / días del mes ≥ 0.80**

Los meses que no cumplan este criterio conservarán su fecha, pero sus valores serán representados como `NaN`.

In [ ]:
# Agregación mensual
temperatura_mensual = temperatura_diaria.resample("ME").agg({
    "temperatura_media_C": "mean",
    "temperatura_min_C": "min",
    "temperatura_max_C": "max"
})

# Número de días válidos disponibles en cada mes
dias_validos = (
    temperatura_diaria["temperatura_media_C"]
    .resample("ME")
    .count()
)

# Número total de días de cada mes
dias_mes = temperatura_mensual.index.days_in_month

# Proporción de datos disponibles
completitud_mensual = dias_validos / dias_mes

# Los meses con menos de 80 % de completitud se consideran incompletos
temperatura_mensual.loc[completitud_mensual < 0.80, :] = float("nan")

temperatura_mensual.head()

Podemos revisar la proporción de días válidos disponibles para cada mes.

Un valor de:

- `1.00` significa que el mes tiene el 100 % de sus días válidos;
- `0.80` significa que dispone del 80 %;
- valores inferiores a `0.80` no cumplen nuestro criterio de completitud.

In [ ]:
completitud_mensual

El DataFrame contiene ahora **una fila por mes**.

Las operaciones utilizadas tienen una interpretación diferente:

- `temperatura_media_C → mean`: promedio de las temperaturas medias diarias del mes;
- `temperatura_min_C → min`: menor temperatura mínima diaria registrada durante el mes;
- `temperatura_max_C → max`: mayor temperatura máxima diaria registrada durante el mes.

Observa también el índice.

Al utilizar `resample("ME")`, Pandas representa cada periodo mensual mediante la fecha correspondiente al **último día del mes**.

Por ejemplo:

`2026-01-31`

representa los datos agregados correspondientes a enero de 2026.

## 11. Comparación entre diferentes resoluciones temporales

A partir de los mismos datos originales hemos construido series con diferentes resoluciones temporales:

**datos originales → horarios → diarios → mensuales**

Cada transformación reduce el número de observaciones, pero facilita el análisis de procesos que ocurren a escalas temporales más amplias.

Comparemos el número de registros de cada serie.

In [ ]:
print("Datos originales:", len(temperatura))
print("Datos horarios:", len(temperatura_horaria))
print("Datos diarios:", len(temperatura_diaria))
print("Datos mensuales:", len(temperatura_mensual))

El número de observaciones disminuye a medida que aumenta la escala temporal.

Los datos originales contienen información detallada sobre las variaciones dentro de cada día. Los datos diarios permiten observar la evolución de la temperatura entre días, mientras que los datos mensuales permiten identificar cambios a escalas temporales más amplias.

La elección de la resolución depende del objetivo del análisis.

Por ejemplo:

- para estudiar el ciclo diario de temperatura → datos de alta frecuencia u horarios;
- para analizar eventos de temperaturas extremas → datos diarios;
- para estudiar variaciones estacionales → datos mensuales.

### Visualizar la temperatura mensual

Representaremos ahora las temperaturas media, mínima y máxima obtenidas para cada mes.

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    temperatura_mensual.index,
    temperatura_mensual["temperatura_media_C"],
    marker="o",
    label="Media"
)

plt.plot(
    temperatura_mensual.index,
    temperatura_mensual["temperatura_min_C"],
    marker="o",
    label="Mínima"
)

plt.plot(
    temperatura_mensual.index,
    temperatura_mensual["temperatura_max_C"],
    marker="o",
    label="Máxima"
)

plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.title("Temperatura mensual – Estación CH01")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

Al comparar las diferentes escalas temporales podemos observar una idea fundamental:

> **La agregación temporal cambia la representación de los datos, pero no genera información nueva.**

Cada valor mensual resume múltiples observaciones diarias, y cada valor diario resume múltiples observaciones de mayor frecuencia.

A medida que agregamos los datos:

- disminuye el número de observaciones;
- se reduce el detalle temporal;
- se facilita la identificación de patrones a escalas temporales mayores.

Por esta razón, antes de realizar una agregación debemos definir qué escala temporal es apropiada para la pregunta que queremos responder.

## 11. Exportar los datos procesados

Hasta ahora hemos trabajado con los datos dentro de Google Colab.

Sin embargo, una vez terminado el procesamiento normalmente necesitamos **guardar los resultados** para utilizarlos posteriormente, compartirlos o incorporarlos en otros análisis.

Durante este notebook generamos tres nuevos conjuntos de datos:

- `temperatura_horaria` → temperatura agregada a escala horaria;
- `temperatura_diaria` → temperatura media, mínima y máxima diaria;
- `temperatura_mensual` → temperatura media, mínima y máxima mensual.

Ahora exportaremos estos DataFrames como archivos **CSV**.

Pandas permite hacerlo mediante la función:

`to_csv()`

In [ ]:
temperatura_horaria.to_csv(
    "CH01_temperatura_horaria.csv"
)

temperatura_diaria.to_csv(
    "CH01_temperatura_diaria.csv"
)

temperatura_mensual.to_csv(
    "CH01_temperatura_mensual.csv"
)

Las tres instrucciones anteriores crean nuevos archivos en el entorno de trabajo de Google Colab.

Observa que no utilizamos `index=False`.

En este caso queremos conservar el índice porque contiene la información temporal de nuestros datos. Por lo tanto, la fecha y hora también se incluirán en los archivos exportados.

Los archivos generados son:

- `CH01_temperatura_horaria.csv`
- `CH01_temperatura_diaria.csv`
- `CH01_temperatura_mensual.csv`

Ahora podemos descargarlos a nuestro computador.

In [ ]:
from google.colab import files

files.download("CH01_temperatura_horaria.csv")
files.download("CH01_temperatura_diaria.csv")
files.download("CH01_temperatura_mensual.csv")

## 12. Resumen

En este notebook trabajamos con datos meteorológicos de alta frecuencia registrados por una estación de monitoreo de bajo costo del Campus Chillán de la Universidad de Concepción.

A partir de los datos originales aprendimos a:

- identificar la resolución temporal de una serie;
- reconocer que los datos reales pueden presentar cambios en su frecuencia de registro;
- trabajar con fecha y hora mediante un `DatetimeIndex`;
- calcular intervalos entre observaciones;
- utilizar `resample()` para realizar agregaciones temporales;
- transformar datos de alta frecuencia a resolución horaria;
- obtener temperaturas media, mínima y máxima diaria;
- obtener temperaturas media, mínima y máxima mensual;
- interpretar cómo cambia la información al modificar la resolución temporal;
- exportar los resultados en formato CSV.


# Actividad: Agregación temporal y visualización de datos meteorológicos

En esta actividad aplicará los procedimientos desarrollados durante la clase utilizando datos de la **estación meteorológica CH01 – Campus Chillán UdeC**.

Cada estudiante trabajará con **una variable meteorológica diferente**.
## Variables

| Variable | Operación de agregación | Unidad |
|---|---|---|
| Precipitación | `sum()` | mm |
| Humedad relativa | `mean()` | % |
| Presión atmosférica | `mean()` | hPa |
| Velocidad del viento | `mean()` | m/s |
| Radiación solar | `mean()` | W/m² |

La operación de agregación depende de la naturaleza de la variable. En el caso de la **precipitación**, utilizaremos `sum()` porque nos interesa conocer la cantidad acumulada durante cada periodo. Para las demás variables utilizaremos `mean()` para representar su valor medio.

## Trabajo a realizar

A partir de los datos originales de la estación:

1. Seleccione únicamente la variable que le fue asignada.
2. Genere una serie **horaria** utilizando la operación de agregación correspondiente.
3. A partir de la serie horaria, genere una serie **diaria**.
4. Considere válido un valor diario únicamente cuando exista al menos **90 % de completitud** de los datos horarios esperados.
5. A partir de la serie diaria, genere una serie **mensual**.
6. Considere válido un valor mensual únicamente cuando exista al menos **90 % de los días válidos esperados para ese mes**.
7. Prepare un gráfico para cada resolución temporal.

Al finalizar deberá tener:

- **Gráfico 1:** serie horaria.
- **Gráfico 2:** serie diaria.
- **Gráfico 3:** serie mensual.

## Requisitos de los gráficos

Los gráficos deben estar preparados para comunicar claramente los resultados y deben incluir:

- título descriptivo;
- nombre de los ejes;
- unidad de la variable;
- fechas legibles;
- leyenda cuando corresponda;
- tamaño de fuente mínimo de **18 pt**;
- líneas y elementos gráficos claramente visibles.

El título debe identificar claramente la **variable, resolución temporal y estación**.

Por ejemplo:

**Precipitación diaria – Estación CH01, Campus Chillán UdeC**

## Entrega

Enviar por correo **únicamente un archivo PDF** con las siguientes características:

- **Tamaño:** A4.
- **Orientación:** vertical.
- **Contenido:** los tres gráficos organizados verticalmente.
- Incluir el **nombre del estudiante** y la **variable asignada**.

Fecha plazo de entrega: Domingo 4 de octubre, 23h59.
